# Phase 3 · Notebook 01 — Pipeline Lite Walkthrough

Pipeline Lite is the cheap, fast variant: detect every span, suppress every DIRECT identifier with a `[TYPE]` tag, leave QUASI mentions intact. This is the right choice for a firm whose threat model is "the LLM provider could log our prompts" rather than "an attacker could re-identify the document by combining its quasi-identifiers".

This notebook walks through the pipeline on a synthetic legal-style paragraph and a real TAB document, then shows the audit log so you can see exactly what the pipeline did and why.

---


## Setup


In [ ]:
import sys
sys.path.insert(0, "../../src")

import json
from anonymisation.data import load_tab
from anonymisation.mapping import SPACY_TO_TAB
from anonymisation.pipeline import LitePipeline

import spacy
print("Loading spaCy en_core_web_trf...")
nlp = spacy.load("en_core_web_trf")

def spacy_predictor(text):
    doc = nlp(text)
    return [(e.start_char, e.end_char, SPACY_TO_TAB[e.label_], e.text)
            for e in doc.ents if e.label_ in SPACY_TO_TAB]

pipeline = LitePipeline(ner_provider=spacy_predictor)


## Synthetic example


In [ ]:
sample = (
    "The applicant, Maria Petrova, is a 47-year-old Bulgarian national living in Plovdiv. "
    "On 12 March 2018 she filed a complaint (Application no. 12345/67) against "
    "the Sofia District Court alleging discrimination on grounds of her Roma ethnicity."
)

result = pipeline(sample)
print("─── ORIGINAL ───")
print(sample)
print("\n─── REDACTED (Lite) ───")
print(result.redacted_text)


## Audit log

Every span the pipeline detected, what it decided to do with it, and why. This is the trail a paralegal could review for a tricky case.


In [ ]:
print(f"{'#':<3} {'TYPE':<10} {'ROLE':<7} {'ACTION':<10} {'TEXT'}")
print("-" * 80)
for i, entry in enumerate(result.audit, 1):
    s = entry.span
    print(f"{i:<3} {s.entity_type:<10} {s.identifier_role:<7} {entry.action:<10} {s.text!r}")


## TAB sample

Same pipeline against a real ECHR document. We use the first 1,500 characters so the output stays readable.


In [ ]:
ds = load_tab()
doc = ds["test"][0]
text = doc["text"][:1500]

result = pipeline(text)
print("─── ORIGINAL (first 600 chars) ───")
print(text[:600], "...")
print("\n─── REDACTED (first 600 chars) ───")
print(result.redacted_text[:600], "...")
print(f"\nTotal spans handled: {len(result.spans)}")
print(f"DIRECT redacted: {sum(1 for e in result.audit if e.action == 'redact')}")
print(f"QUASI left intact: {sum(1 for e in result.audit if e.action == 'leave')}")


## Caveats — what Lite does *not* do

- **Doesn't touch QUASI mentions.** Every date, location, demographic descriptor and quantity is left exactly as it was. If the threat model is mosaic re-identification, this is not enough — see Notebook 02 (Pipeline Pro).
- **Trusts the NER classifier's role decisions.** Pipeline Lite uses the conservative defaults in `roles.py`: PERSON, ORG, CODE, MISC are always DIRECT; DATETIME, LOC, DEM, QUANTITY are always QUASI. A real deployment would let the firm override these per matter type.
- **Doesn't detect document-level identifiers.** Layout cues like letterhead, page footers, or a docket number repeated on every page are out of scope here.

If those caveats matter, Pipeline Pro is the appropriate variant. Otherwise, Lite is the simpler, faster, more predictable choice.
